# ViNet-подобная архитектура

## Сборка picle

In [ ]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import pickle

DATA_PATH = "dataset/run_1"
IMG_PATH = os.path.join(DATA_PATH, "images")

# подгружаю данные полетов (IMU и GT позы)
imu_df = pd.read_csv(os.path.join(DATA_PATH, "imu.csv"))
pose_df = pd.read_csv(os.path.join(DATA_PATH, "poses.csv"))

imu_t = imu_df["t"].values
pose_t = pose_df["t"].values

# подгружаю изображения
img_files = sorted(glob.glob(os.path.join(IMG_PATH, "*.png")))

img_times = np.array([
    float(os.path.basename(f).replace(".png", ""))
    for f in img_files
])

def nearest_idx(arr, t):
    return np.argmin(np.abs(arr - t))

# формирую датасет (окно из двух кадров t, t -1, и данные IMU между ними)
data = []

for i in range(1, len(img_files)):

    t0 = img_times[i-1]
    t1 = img_times[i]

    # === изображение ===
    img = cv2.imread(img_files[i])
    img = cv2.resize(img, (224,224))
    img = img.astype(np.float32) / 255.0
    img = np.transpose(img, (2,0,1))  # (3,H,W)

    # IMU между кадрами 
    mask = (imu_t >= t0) & (imu_t <= t1)
    imu_seq = imu_df.loc[mask, ["ax","ay","az","gx","gy","gz"]].values

    imu_seq[:, 2] *= -1  # az
    imu_seq[:, 5] *= -1  # gz
    
    if len(imu_seq) < 5:
        continue

    # позы
    idx0 = nearest_idx(pose_t, t0)
    idx1 = nearest_idx(pose_t, t1)

    p0_raw = pose_df.iloc[idx0][["x","y","z"]].values
    p1_raw = pose_df.iloc[idx1][["x","y","z"]].values
    
    #  тут решил переводить систему координат из NED → ENU для удобства интерпритации
    p0 = np.array([p0_raw[0], p0_raw[1], -p0_raw[2]])
    p1 = np.array([p1_raw[0], p1_raw[1], -p1_raw[2]])

    delta = p1 - p0

    data.append({
        "img_t": img,
        "imu_seq": imu_seq,
        "delta_pose": delta
    })

print("Сэмплов:", len(data))

with open(os.path.join(DATA_PATH, "dataset.pkl"), "wb") as f:
    pickle.dump(data, f)

print("dataset.pkl сохранён")

## Датасет

In [ ]:
import pickle
from torch.utils.data import DataLoader

with open("dataset/run_1/dataset.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
print(data[0]["img_t"].shape)

In [ ]:
import torch
from torch.utils.data import Dataset
import numpy as np

class VIOSequenceDataset(Dataset):
    def __init__(self, data, seq_len=5, imu_len=20):
        self.data = data
        self.seq_len = seq_len
        self.imu_len = imu_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def pad_imu(self, imu):
        if len(imu) >= self.imu_len:
            return imu[:self.imu_len]
        else:
            pad = np.zeros((self.imu_len - len(imu), 6))
            return np.vstack([imu, pad])

    def __getitem__(self, idx):
        img_seq = []
        imu_seq = []
        delta_seq = []

        for i in range(self.seq_len):
            item = self.data[idx + i]

            img_seq.append(item["img_t"])
            imu_seq.append(self.pad_imu(item["imu_seq"]))
            delta_seq.append(item["delta_pose"][:3])

        img_seq = torch.tensor(np.array(img_seq), dtype=torch.float32)  # (T,3,224,224)
        imu_seq = torch.tensor(np.array(imu_seq), dtype=torch.float32)  # (T,imu_len,6)
        delta_seq = torch.tensor(np.array(delta_seq), dtype=torch.float32)  # (T,3)

        return img_seq, imu_seq, delta_seq

## Модель

In [ ]:
import torch.nn as nn
import torchvision.models as models

class VIONetV2(nn.Module):
    def __init__(self):
        super().__init__()

        # CNN слой
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])

        # IMU энкодер
        self.imu_lstm = nn.LSTM(6, 64, batch_first=True)

        # фьюжн
        self.fusion_fc = nn.Linear(512 + 64, 256)

        # LSTM
        self.temporal_lstm = nn.LSTM(256, 256, batch_first=True)

        # выход
        self.head = nn.Linear(256, 3)  # velocity 

    def forward(self, img_seq, imu_seq):
        B, T, C, H, W = img_seq.shape

        # CNN блок
        img_seq = img_seq.view(B*T, C, H, W)
        img_feat = self.cnn(img_seq).view(B, T, 512)

        fused_seq = []

        for t in range(T):
            imu_t = imu_seq[:, t]  # (B, imu_len, 6)
            _, (h, _) = self.imu_lstm(imu_t)
            imu_feat = h[-1]

            x = torch.cat([img_feat[:, t], imu_feat], dim=1)
            x = self.fusion_fc(x)
            fused_seq.append(x)

        fused_seq = torch.stack(fused_seq, dim=1)

        # lstm
        out, _ = self.temporal_lstm(fused_seq)

        vel = self.head(out)  # (B,T,3)

        return vel

## Loss-function

In [ ]:
import torch

def trajectory_from_velocity(vel):
    return torch.cumsum(vel, dim=1)  # (B,T,3)

def loss_fn(pred_vel, gt_delta):
    # локальный лосс
    loss_local = ((pred_vel - gt_delta) ** 2).mean()

    # общий лосс по траектории
    pred_traj = trajectory_from_velocity(pred_vel)
    gt_traj   = trajectory_from_velocity(gt_delta)

    loss_traj = ((pred_traj - gt_traj) ** 2).mean()

    # итоговый лосс
    return loss_local + 0.5 * loss_traj

## Обучение

In [ ]:
# заношу датасет и лоадер
dataset = VIOSequenceDataset(data, seq_len=5, imu_len=20)

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = VIONetV2().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for img_seq, imu_seq, delta_seq in loader:
        img_seq = img_seq.to(device)
        imu_seq = imu_seq.to(device)
        delta_seq = delta_seq.to(device)

        pred_vel = model(img_seq, imu_seq)

        loss = loss_fn(pred_vel, delta_seq)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: loss = {total_loss:.4f}")

## Инференс


In [ ]:
import numpy as np

model.eval()

pred_traj = []
gt_traj = []

pred_pos = np.zeros(3)
gt_pos   = np.zeros(3)

SEQ_LEN = 5

for i in range(len(dataset)):
    img_seq, imu_seq, delta_seq = dataset[i]

    img_seq = img_seq.unsqueeze(0).to(device)
    imu_seq = imu_seq.unsqueeze(0).to(device)

    with torch.no_grad():
        pred_vel = model(img_seq, imu_seq)[0].cpu().numpy()  # (T,3)

    gt_vel = delta_seq.numpy()

    # тут интегрирую скорости для траектории полета
    for t in range(len(pred_vel)):
        pred_pos += pred_vel[t]
        gt_pos   += gt_vel[t]

        pred_traj.append(pred_pos.copy())
        gt_traj.append(gt_pos.copy())

pred_traj = np.array(pred_traj)
gt_traj = np.array(gt_traj)

## Визуализация

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(10,7))
ax = fig.add_subplot(111, projection='3d')

ax.plot(
    gt_traj[:,0],
    gt_traj[:,1],
    gt_traj[:,2],
    label="GT",
    linewidth=2
)

ax.plot(
    pred_traj[:,0],
    pred_traj[:,1],
    pred_traj[:,2],
    label="Pred",
    linewidth=2
)

ax.set_title("3D Траектория")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")

ax.legend()
plt.show()

## Сохранение модели

In [ ]:
SAVE_PATH = "model_vio.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": EPOCHS
}, SAVE_PATH)

print("сохранена:", SAVE_PATH)

## Загрузка модели

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = VIONetV2().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

checkpoint = torch.load("model_vio.pth", map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

model.eval()

print("Модель загружена")